In [1]:
# Training

In [2]:
import numpy as np
import os
import glob
import cv2
import copy
import json

import torch.utils.data
import torchvision.models.segmentation
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import ToTensor
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

import imgaug as ia
import imgaug.augmenters as iaa
from imgaug.augmentables.segmaps import SegmentationMapsOnImage

ia.seed(2)
random_state = np.random.RandomState(42)

In [3]:
img_dir = 'training_images/'
gt_dir = 'ground_truth/'
gt_save_dir = 'gt_separated/'
img_list = glob.glob(img_dir + '*/*.png')
gt_list = glob.glob(gt_dir + '*/*.png')

imageSize=[600,600]

In [4]:
def separateMask():
    for gt_item in gt_list:
        gt_mask = cv2.imread(gt_item)
        gt_mask = np.max(gt_mask, axis=2)
        num_labels, label_img = cv2.connectedComponents(gt_mask)
        for label in range(1, num_labels):
            mask = np.zeros(label_img.shape, dtype=np.uint8)
            mask[label_img == label] = 255
            root_ext = os.path.splitext(gt_item)
            new_path = root_ext[0].replace('ground_truth', 'gt_separated')
            new_path = new_path.split('_WH')[0]
            if not os.path.exists(new_path):
                os.mkdir(new_path)
            mask_filename = new_path + '/' + str(label) + '.png'
            cv2.imwrite(mask_filename, mask)

In [5]:
# separateMask()

In [6]:
def train_valid_split(img_list, random_state, proportion=9):
    train_img_list = []
    valid_img_list = []
    
    valid_select_vector = np.zeros(len(img_list)).astype('int')
    no_of_training_imgs = int(len(img_list)/proportion)
    
    k = 0
    for i in range(no_of_training_imgs):
        foo = random_state.randint(0, proportion)
        valid_select_vector[foo + k] = 1
        k += proportion
    
    for img_no in range(len(img_list)):
        if valid_select_vector[img_no] == 0:
            train_img_list.extend([img_list[img_no]])
        else:
            valid_img_list.extend([img_list[img_no]])
    
    return train_img_list, valid_img_list

In [7]:
augmentation_pipeline_train = iaa.Sequential([
    iaa.Sometimes(0.5, iaa.ElasticTransformation(alpha=100, sigma=10)),
    iaa.Sometimes(0.5, iaa.Fliplr(1)),
    iaa.Sometimes(0.5, iaa.GaussianBlur(sigma=(0.0, 2.0))),
    iaa.Sometimes(0.5, iaa.Affine(scale=(0.5, 2.0), translate_percent={"x": (-0.15, 0.15), "y": (-0.15, 0.15)}, rotate=(-45, 45))),
    iaa.Sometimes(0.5, iaa.PerspectiveTransform(scale=(0.01, 0.15))),
    iaa.Sometimes(0.5, iaa.ChangeColorTemperature((4500, 9500))),
    iaa.Sometimes(0.5, iaa.GammaContrast((0.75, 1.25)))
])

In [8]:
class GVHD_Dataset(Dataset):
    def __init__(self, img_dir, mode, transform=None):
        self.transform = transform 
        self.img_dir = img_dir
        self.mode = mode
        
    def __len__(self):
        size=len(self.img_dir)
        return size
    
    def __getitem__(self, idx):
        img_path = self.img_dir[idx]
        image = cv2.imread(img_path)
        
        root_ext = os.path.splitext(img_path)
        gt_dir = root_ext[0].replace('training_images', 'gt_separated')
        masks = []
        for gt_name in os.listdir(gt_dir):
            les_mask = cv2.imread(gt_dir + '/' + gt_name, 0)
            les_mask = (les_mask > 0).astype(np.uint8)
            masks.append(les_mask)
        
        boxes = torch.zeros([len(masks), 4], dtype=torch.float32)
        if self.mode == 'train':
            segmaps = []
            for mask in masks:
                segmaps.append(SegmentationMapsOnImage(mask, shape=image.shape))
#             image_aug, segmap_aug = augmentation_pipeline_train(image=image, segmentation_maps=segmaps)
            
            image = cv2.resize(image, imageSize, cv2.INTER_LINEAR)
            image = torch.as_tensor(image, dtype=torch.float32)
            i = 0
            masks_copy = copy.deepcopy(masks)
            for mask_c in masks_copy:
                if np.sum(cv2.resize(mask_c, imageSize,cv2.INTER_NEAREST)) == 0:
                    del masks[i]
                    tmp_boxes = boxes
                    boxes = torch.zeros([len(masks), 4], dtype=torch.float32)
                    boxes = tmp_boxes[0:len(masks), 0:4]
                else:
                    masks[i] = cv2.resize(mask_c, imageSize,cv2.INTER_NEAREST)
                    x,y,w,h = cv2.boundingRect(masks[i])
                    boxes[i] = torch.tensor([x, y, x+w, y+h])
                    i += 1

        else:
            image = cv2.resize(image, imageSize, cv2.INTER_LINEAR)
            image = torch.as_tensor(image, dtype=torch.float32)
            i = 0
            masks_copy = copy.deepcopy(masks)
            for mask_c in masks_copy:
                if np.sum(cv2.resize(mask_c, imageSize,cv2.INTER_NEAREST)) == 0:
                    del masks[i]
                    tmp_boxes = boxes
                    boxes = torch.zeros([len(masks), 4], dtype=torch.float32)
                    boxes = tmp_boxes[0:len(masks), 0:4]
                else:
                    masks[i] = cv2.resize(mask_c, imageSize,cv2.INTER_NEAREST)
                    x,y,w,h = cv2.boundingRect(masks[i])
                    boxes[i] = torch.tensor([x, y, x+w, y+h])
                    i += 1
                
        masks = torch.as_tensor(masks, dtype=torch.uint8)

        data = {}
        data["boxes"] = boxes
        data["labels"] = torch.ones((len(masks),), dtype=torch.int64)   
        data["masks"] = masks
        
        return image, data

In [9]:
def cGVHD_UNet_training(data_train,
                        data_valid,
                        save_root='.' + os.sep,
                        save_name='cGVHD_MaskRCNN_',
                        total_epochs = 50,
                        lr_1 = 1e-5,
                        train_batchsize = 4,
                        valid_batchsize = 1
                       ):
    torch.cuda.empty_cache()

    print('Training ' + save_name + ' for ' + str(total_epochs) + ' epochs...')

    train_loader = DataLoader(data_train, batch_size=train_batchsize, shuffle=True, collate_fn=lambda x: x )
    valid_loader = DataLoader(data_valid, batch_size=valid_batchsize, shuffle=False, collate_fn=lambda x: x )
    
    device = torch.device('cuda')

    model = torchvision.models.detection.maskrcnn_resnet50_fpn(pretrained=True)  # load an instance segmentation model pre-trained pre-trained on COCO
    in_features = model.roi_heads.box_predictor.cls_score.in_features  # get number of input features for the classifier
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features,num_classes=2)  # replace the pre-trained head with a new one
    model.to(device)# move model to the right device
    
    optimizer = torch.optim.AdamW(params=model.parameters(), lr=lr_1)
    model.train()

    loss_set = []
    for epoch_no in range(total_epochs):
        trainset_iter = iter(train_loader)
        while True:
            try:
                packed_data = next(trainset_iter)
                batch_Imgs = []
                batch_Data = []
                for i in range(len(packed_data)):
                    curr_load = packed_data[i]
                    batch_Imgs.append(curr_load[0])
                    batch_Data.append(curr_load[1])
                batch_Imgs = torch.stack([torch.as_tensor(d) for d in batch_Imgs],0)
                batch_Imgs = batch_Imgs.swapaxes(1, 3).swapaxes(2, 3)
                batch_Imgs = list(image.to(device) for image in batch_Imgs)
                batch_Data = [{k: v.to(device) for k, v in t.items()} for t in batch_Data]
                optimizer.zero_grad()
                loss_dict = model(batch_Imgs, batch_Data)
                losses = sum(loss for loss in loss_dict.values())
                losses.backward()
                optimizer.step()
            except StopIteration:
                break
        
        validset_iter = iter(valid_loader)
        while True:
            try:
                packed_data = next(validset_iter)
                batch_Imgs = []
                batch_Data = []
                for i in range(valid_batchsize):
                    curr_load = packed_data[i]
                    batch_Imgs.append(curr_load[0])
                    batch_Data.append(curr_load[1])
                batch_Imgs = torch.stack([torch.as_tensor(d) for d in batch_Imgs],0)
                batch_Imgs = batch_Imgs.swapaxes(1, 3).swapaxes(2, 3)
                batch_Imgs = list(image.to(device) for image in batch_Imgs)
                batch_Data = [{k: v.to(device) for k, v in t.items()} for t in batch_Data]
                
                loss_dict = model(batch_Imgs, batch_Data)
                losses = sum(loss for loss in loss_dict.values())
                loss_set.append(losses.item())
                print('epoch', str(epoch_no), 'loss:', losses.item())
            except StopIteration:
                break
        
        if epoch_no % 5 == 0:
            torch.save(model.state_dict(), str(epoch_no)+".torch")
        print('epoch finished, no: ', epoch_no)
        
    with open('training_performance.json', 'w') as fp:
        json.dump(loss_set, fp, indent=4)

In [10]:
# Create train and validation file lists, and prepare data loaders
train_img_list, valid_img_list = train_valid_split(img_list, random_state, 10)

dataset_train = GVHD_Dataset(train_img_list,'train')
dataset_valid = GVHD_Dataset(valid_img_list,'valid')

cGVHD_UNet_training(dataset_train, dataset_valid,
                    train_batchsize = 4,
                    total_epochs = 40,
                   )

Training cGVHD_MaskRCNN_ for 40 epochs...


C:\Users\jiangb\AppData\Local\Temp\ipykernel_7900\3722373330.py:63: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at  C:\cb\pytorch_1000000000000\work\torch\csrc\utils\tensor_new.cpp:210.)
  masks = torch.as_tensor(masks, dtype=torch.uint8)


epoch 0 loss: 30.27175521850586
epoch 0 loss: 42.07350158691406
epoch 0 loss: 41.300445556640625
epoch 0 loss: 31.80811309814453
epoch 0 loss: 47.370811462402344
epoch 0 loss: 39.668453216552734
epoch finished, no:  0
epoch 1 loss: 13.955780982971191
epoch 1 loss: 17.62430763244629
epoch 1 loss: 16.89060401916504
epoch 1 loss: 15.211343765258789
epoch 1 loss: 17.843931198120117
epoch 1 loss: 17.9492244720459
epoch finished, no:  1
epoch 2 loss: 11.03481674194336
epoch 2 loss: 11.537229537963867
epoch 2 loss: 12.483328819274902
epoch 2 loss: 10.454353332519531
epoch 2 loss: 13.884349822998047
epoch 2 loss: 10.898744583129883
epoch finished, no:  2


KeyboardInterrupt: 